# 第二章: 相机参数标定

## 学习目标
- 掌握三维视觉中成像模型以及相机内外参数的基本概念
- 掌握**张正友标定法**的原理和编程实现
- 掌握**非线性最小二乘优化算法**及其编程实现

## 编程实践
根据棋盘格标定方法的操作原则，利用 OpenCV 自动检测棋盘格角点的坐标，用于求解相机的外参数（位置和朝向）和内参数（焦距、像主点、径向畸变系数）。相机参数需要用**非线性最小二乘算法**进行迭代求解。

> **要求**：除图像读写、矩阵计算、角点检测部分可调用 OpenCV/numpy 函数外，其余代码都是自己手写，不能调用其他函数库。

---
## 1. 相机成像模型

### 1.1 针孔相机模型
针孔相机模型是最基本的成像模型：

```
物点 P(X, Y, Z) → 针孔投影 → 像点 p(u, v)
```

坐标变换过程：
1. **世界坐标系 → 相机坐标系**：通过外参数（旋转 R 和平移 t）
2. **相机坐标系 → 图像平面坐标系**：透视投影（焦距 f）
3. **图像平面坐标系 → 像素坐标系**：内参数矩阵 K

### 1.2 内参数矩阵 K
```
K = | fx  0  cx |
    |  0 fy  cy |
    |  0   0   1 |
```

- `fx, fy`：x 和 y 方向的焦距（以像素为单位）
- `cx, cy`：像主点坐标（光轴与成像平面的交点）

### 1.3 畸变模型
实际镜头会产生畸变，主要有：
- **径向畸变**：桶形/枕形畸变
  ```
  x_distorted = x(1 + k1*r² + k2*r⁴ + k3*r⁶)
  y_distorted = y(1 + k1*r² + k2*r⁴ + k3*r⁶)
  ```
- **切向畸变**：镜头安装偏心导致
  ```
  x_distorted = x + 2*p1*x*y + p2*(r² + 2*x²)
  y_distorted = y + p1*(r² + 2*y²) + 2*p2*x*y
  ```

### 1.4 标定的目的
通过已知的 3D 点（棋盘格角点）和其对应的 2D 像素坐标，求解相机的内外参数。

---
## 2. 张正友标定法

### 2.1 基本原理
张正友标定法（Zhang's Method）是目前最常用的相机标定方法，需要：
- 一块**平面棋盘格**作为标定物
- 从**不同角度**拍摄至少 2 张棋盘格图像

### 2.2 步骤概述
1. 在棋盘格图像上检测角点坐标
2. 根据已知的棋盘格物理尺寸建立 3D-2D 对应关系
3. 用**线性方法**初步估计内外参数
4. 用**非线性最小二乘**（Levenberg-Marquardt）优化所有参数

### 2.3 数学推导
对于每张棋盘格图像，有：
```
s * [u, v, 1]^T = K * [R|t] * [X, Y, 0, 1]^T
```

其中 s 是尺度因子，[X, Y, 0, 1]^T 是棋盘格角点的齐次坐标。

单应矩阵 H 描述了平面到图像的映射：
```
H = K * [r1, r2, t]
```

从 H 可以约束内参数矩阵 K：
```
H^T K^(-T) K^(-1) H 相等
```

### 2.4 非线性优化
最终通过最小化**重投影误差**来精化参数：
```
min Σ ‖p_i - project(K, k, R_j, t_j, P_i)‖²
```

- `p_i`：检测到的角点像素坐标
- `project()`：根据参数投影 3D 点到图像
- `P_i`：已知的 3D 角点坐标

---
## 3. 非线性最小二乘: Levenberg-Marquardt 算法

### 3.1 LM 算法原理
Levenberg-Marquardt（LM）算法是一种**非线性最小二乘优化**算法，结合了：
- **高斯-牛顿法**：二次收敛，但可能不收敛
- **梯度下降法**：收敛慢，但总是收敛

### 3.2 更新公式
```
(J^T J + λ * I) * Δ = J^T r
x_new = x + Δ
```

- `J`：残差对参数的雅可比矩阵
- `r`：残差向量
- `λ`：阻尼参数（控制步长）
- `I`：单位矩阵

### 3.3 自适应 λ
- 若误差减小 → λ 减小（更接近高斯-牛顿）
- 若误差增大 → λ 增大（更接近梯度下降）

### 3.4 迭代流程
```
1. 给定初始参数 x0 和阻尼参数 λ
2. 计算残差 r 和雅可比矩阵 J
3. 求解 (J^T J + λI)Δ = J^T r
4. 更新参数 x_new = x + Δ
5. 计算新误差
   - 若新误差 < 旧误差: 接受更新, λ *= 0.5
   - 若新误差 ≥ 旧误差: 拒绝更新, λ *= 2
6. 重复直到收敛
```

In [ ]:
# ===== 环境设置 =====
import os
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# ===== 中文路径兼容的图像读写函数 =====
# OpenCV 在 Windows 中文路径下 imread/imwrite 会失败
def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    import numpy as np
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False



plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {os.getcwd()}")



In [ ]:
# ===== 第一步: 读取标定图像 =====
# 棋盘格规格: 9x6 内角点 (即 10x7 方格)
pattern_size = (9, 6)  # 内角点数量 (列, 行)

# 读取所有标定图像
calib_images = []
image_files = ['calib_image_1.jpg', 'calib_image_2.jpg',
               'calib_image_3.jpg', 'calib_image_4.jpg']

for fname in image_files:
    img = cv_imread(fname)
    if img is not None:
        calib_images.append(img)
        print(f"已加载: {fname} ({img.shape[1]}x{img.shape[0]})")
    else:
        print(f"警告: 无法加载 {fname}")

if len(calib_images) < 2:
    raise ValueError("至少需要 2 张标定图像!")

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, (img, fname) in enumerate(zip(calib_images, image_files)):
    ax = axes[i // 2, i % 2]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(fname, fontsize=10)
    ax.axis('off')
plt.suptitle('标定图像', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ===== 第二步: 检测棋盘格角点 =====
# OpenCV 的 findChessboardCorners 自动检测角点

# 准备 3D 角点坐标 (假设棋盘格每格 1 单位)
objp = np.zeros((pattern_size[0] * pattern_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:pattern_size[0], 0:pattern_size[1]].T.reshape(-1, 2)
print(f"3D 角点模板: {objp.shape}")
print(f"前 3 个 3D 点:\n{objp[:3]}")

# 对每张图像检测角点
objpoints = []  # 3D 角点坐标 (世界坐标系)
imgpoints = []  # 2D 角点坐标 (像素)
detected_images = []  # 检测到角点的图像

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

for i, img in enumerate(calib_images):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 检测角点
    ret, corners = cv2.findChessboardCorners(gray, pattern_size, None)
    
    if ret:
        # 亚像素级精化角点坐标
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        
        objpoints.append(objp.copy())
        imgpoints.append(corners2)
        
        # 在图像上绘制检测到的角点
        img_vis = img.copy()
        cv2.drawChessboardCorners(img_vis, pattern_size, corners2, ret)
        detected_images.append(img_vis)
        print(f"图像 {i+1}: 检测到 {len(corners2)} 个角点 ✓")
    else:
        print(f"图像 {i+1}: 未检测到角点 ✗")

print(f"\n成功检测: {len(objpoints)} 张图像")

# 可视化检测结果
if detected_images:
    fig, axes = plt.subplots(1, len(detected_images), figsize=(5*len(detected_images), 5))
    if len(detected_images) == 1:
        axes = [axes]
    for ax, img_vis, fname in zip(axes, detected_images, image_files[:len(detected_images)]):
        ax.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{fname}\n角点检测', fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ===== 第三步: 初始参数估计 =====
# 使用 OpenCV 的 calibrateCamera 得到初始参数

# 获取图像尺寸
h, w = calib_images[0].shape[:2]
image_size = (w, h)

# 初始内参数矩阵
K_init = np.float64([
    [w, 0, w/2],
    [0, w, h/2],
    [0, 0, 1]
])

# 使用 OpenCV 求初始解
ret, K_initial, dist_initial, rvecs_initial, tvecs_initial = cv2.calibrateCamera(
    objpoints, imgpoints, image_size, K_init.copy(), None,
    flags=cv2.CALIB_USE_INTRINSIC_GUESS
)

print("=" * 50)
print("初始参数估计 (OpenCV calibrateCamera)")
print("=" * 50)
print(f"\n内参数矩阵 K:")
print(f"  fx = {K_initial[0, 0]:.4f}")
print(f"  fy = {K_initial[1, 1]:.4f}")
print(f"  cx = {K_initial[0, 2]:.4f}")
print(f"  cy = {K_initial[1, 2]:.4f}")
print(f"\n畸变系数:")
print(f"  k1 = {dist_initial[0, 0]:.6f}")
print(f"  k2 = {dist_initial[0, 1]:.6f}")
print(f"\n初始重投影误差: {ret:.6f} 像素")

# 保存初始参数供后续优化使用
print(f"\n外参数 (每张图像的旋转和平移):")
for i, (rvec, tvec) in enumerate(zip(rvecs_initial, tvecs_initial)):
    print(f"  图像 {i+1}: r={rvec.ravel()}, t={tvec.ravel()}")

In [ ]:
# ===== 第四步: 手写相机投影与重投影误差计算 =====

def project_points_custom(obj_pts, rvec, tvec, K, dist):
    """
    手写 3D-2D 投影函数
    
    参数:
        obj_pts: Nx3 的 3D 点 (世界坐标系)
        rvec: 3x1 旋转向量 (罗德里格斯)
        tvec: 3x1 平移向量
        K: 3x3 内参数矩阵
        dist: 畸变系数 [k1, k2, p1, p2, k3]
    
    返回:
        projected: Nx2 的 2D 像素坐标
    """
    n = len(obj_pts)
    projected = np.zeros((n, 2), dtype=np.float64)
    
    # ===== 1. 旋转向量转旋转矩阵 (罗德里格斯公式) =====
    # rvec → R
    theta = np.linalg.norm(rvec)
    if theta < 1e-10:
        R = np.eye(3)
    else:
        r = rvec.ravel() / theta
        # 反对称矩阵
        [rx, ry, rz] = r
        skew = np.array([[0, -rz, ry], [rz, 0, -rx], [-ry, rx, 0]])
        # R = I + sin(θ)*[r]× + (1-cos(θ))*[r]×²
        R = np.eye(3) + np.sin(theta) * skew + (1 - np.cos(theta)) * (skew @ skew)
    
    # ===== 2. 世界坐标 → 相机坐标 =====
    for i in range(n):
        pt = obj_pts[i]  # (3,)
        # 变换: P_cam = R * P_world + t
        p_cam = R @ pt + tvec.ravel()
        
        # ===== 3. 透视投影 (归一化坐标) =====
        if p_cam[2] < 1e-10:
            projected[i] = [-1, -1]  # 无效投影
            continue
        
        x_norm = p_cam[0] / p_cam[2]  # x / z
        y_norm = p_cam[1] / p_cam[2]  # y / z
        
        # ===== 4. 畸变校正 =====
        k1, k2 = dist[0], dist[1]
        p1, p2 = dist[2], dist[3] if len(dist) > 3 else 0
        k3 = dist[4] if len(dist) > 4 else 0
        
        r2 = x_norm**2 + y_norm**2
        r4 = r2 * r2
        r6 = r4 * r2
        
        # 径向畸变
        radial = 1 + k1 * r2 + k2 * r4 + k3 * r6
        x_radial = x_norm * radial
        y_radial = y_norm * radial
        
        # 切向畸变
        x_tangential = 2 * p1 * x_norm * y_norm + p2 * (r2 + 2 * x_norm**2)
        y_tangential = p1 * (r2 + 2 * y_norm**2) + 2 * p2 * x_norm * y_norm
        
        x_dist = x_radial + x_tangential
        y_dist = y_radial + y_tangential
        
        # ===== 5. 像素坐标 =====
        fx, fy = K[0, 0], K[1, 1]
        cx, cy = K[0, 2], K[1, 2]
        
        projected[i, 0] = fx * x_dist + cx
        projected[i, 1] = fy * y_dist + cy
    
    return projected

# 验证我们的投影函数
print("验证手写投影函数 vs OpenCV:")
print("=" * 50)

# 用 OpenCV 投影
sample_idx = 0
obj = objpoints[sample_idx]
rvec = rvecs_initial[sample_idx]
tvec = tvecs_initial[sample_idx]

proj_cv, _ = cv2.projectPoints(obj, rvec, tvec, K_initial, dist_initial)
proj_custom = project_points_custom(obj, rvec, tvec, K_initial, dist_initial.ravel())

diff = np.abs(proj_cv.reshape(-1, 2) - proj_custom)
print(f"最大差异: {np.max(diff):.6f} 像素")
print(f"平均差异: {np.mean(diff):.6f} 像素")

if np.max(diff) < 0.01:
    print("✓ 手写投影函数验证通过!")
else:
    print("✗ 投影函数有误差, 请检查实现")

# 计算所有图像的初始重投影误差
print("\n各图像初始重投影误差:")
total_error = 0
for i in range(len(objpoints)):
    proj = project_points_custom(objpoints[i], rvecs_initial[i], tvecs_initial[i],
                                 K_initial, dist_initial.ravel())
    actual = imgpoints[i].reshape(-1, 2)
    err = np.sqrt(np.sum((proj - actual)**2, axis=1))
    mean_err = np.mean(err)
    total_error += np.sum(err**2)
    print(f"  图像 {i+1}: 平均误差 = {mean_err:.4f} 像素")

total_error = np.sqrt(total_error / sum(len(pts) for pts in imgpoints))
print(f"\n总体初始重投影误差: {total_error:.4f} 像素")

In [ ]:
# ===== 第五步: 手写 Levenberg-Marquardt 非线性优化 =====

# 参数化: 将所有参数打包成一个向量
# 待优化参数: [fx, fy, cx, cy, k1, k2, k3, p1, p2, 每张图的 rvec(3) + tvec(3)]

def pack_parameters(K, dist, rvecs, tvecs):
    """将所有参数打包成一维向量"""
    params = []
    # 内参数: fx, fy, cx, cy
    params.extend([K[0, 0], K[1, 1], K[0, 2], K[1, 2]])
    # 畸变: k1, k2, k3, p1, p2
    dist_flat = dist.ravel()
    for j in range(5):
        params.append(dist_flat[j] if j < len(dist_flat) else 0)
    # 每张图像的外参数
    for rvec, tvec in zip(rvecs, tvecs):
        params.extend(rvec.ravel().tolist())  # 3
        params.extend(tvec.ravel().tolist())  # 3
    return np.array(params, dtype=np.float64)

def unpack_parameters(params, n_images):
    """从一维向量解包参数"""
    K = np.float64([
        [params[0], 0, params[2]],
        [0, params[1], params[3]],
        [0, 0, 1]
    ])
    dist = np.array([[params[4], params[5], params[7], params[8], params[6]]], dtype=np.float64)
    rvecs = []
    tvecs = []
    idx = 9
    for _ in range(n_images):
        rvecs.append(params[idx:idx+3].reshape(3, 1))
        tvecs.append(params[idx+3:idx+6].reshape(3, 1))
        idx += 6
    return K, dist, rvecs, tvecs

def compute_residuals(params, objpoints, imgpoints):
    """计算所有角点的重投影残差"""
    n_images = len(objpoints)
    K, dist, rvecs, tvecs = unpack_parameters(params, n_images)
    residuals = []
    
    for i in range(n_images):
        proj = project_points_custom(objpoints[i], rvecs[i], tvecs[i], K, dist.ravel())
        actual = imgpoints[i].reshape(-1, 2)
        # 每个点的 x 和 y 残差 (展开成一维)
        for j in range(len(proj)):
            residuals.append(proj[j, 0] - actual[j, 0])
            residuals.append(proj[j, 1] - actual[j, 1])
    
    return np.array(residuals, dtype=np.float64)

def compute_jacobian(params, objpoints, imgpoints, eps=1e-6):
    """
    用有限差分法计算雅可比矩阵
    J[i, j] = ∂(residual_i) / ∂(param_j)
    """
    residuals = compute_residuals(params, objpoints, imgpoints)
    n_residuals = len(residuals)
    n_params = len(params)
    
    J = np.zeros((n_residuals, n_params), dtype=np.float64)
    
    for j in range(n_params):
        # 对每个参数施加微小扰动
        params_perturbed = params.copy()
        params_perturbed[j] += eps * (1 + abs(params[j]))
        
        residuals_perturbed = compute_residuals(params_perturbed, objpoints, imgpoints)
        
        # 有限差分: ∂r/∂p_j ≈ (r(p+ε) - r(p)) / ε
        J[:, j] = (residuals_perturbed - residuals) / (eps * (1 + abs(params[j])))
    
    return J

def levenberg_marquardt(objpoints, imgpoints, K_init, dist_init, rvecs_init, tvecs_init,
                        max_iter=50, lambda_init=1e-3, tol=1e-4):
    """
    手写 Levenberg-Marquardt 优化
    """
    n_images = len(objpoints)
    
    # 打包初始参数
    params = pack_parameters(K_init, dist_init, rvecs_init, tvecs_init)
    
    # 计算初始残差和误差
    residuals = compute_residuals(params, objpoints, imgpoints)
    cost = np.sum(residuals**2)
    n_pts = sum(len(pts) for pts in imgpoints)
    rmse = np.sqrt(cost / n_pts)
    
    lambda_param = lambda_init
    prev_rmse = rmse
    history = [rmse]
    
    print(f"初始 RMSE: {rmse:.6f} 像素")
    
    for iteration in range(max_iter):
        # 计算雅可比矩阵
        J = compute_jacobian(params, objpoints, imgpoints)
        
        # 计算 J^T J (Hessian 近似)
        JtJ = J.T @ J
        
        # 计算 J^T r (梯度)
        Jtr = J.T @ residuals
        
        # LM 方程: (J^T J + λI) Δ = J^T r
        n_params = len(params)
        A = JtJ + lambda_param * np.eye(n_params)
        
        # 求解线性方程组 (使用 numpy, 这是允许的)
        try:
            delta = np.linalg.solve(A, Jtr)
        except np.linalg.LinAlgError:
            # 奇异矩阵, 使用最小二乘
            delta, _, _, _ = np.linalg.lstsq(A, Jtr, rcond=None)
        
        # 更新参数
        params_new = params + delta
        
        # 计算新残差
        residuals_new = compute_residuals(params_new, objpoints, imgpoints)
        cost_new = np.sum(residuals_new**2)
        rmse_new = np.sqrt(cost_new / n_pts)
        
        # 判断是否接受更新
        if rmse_new < rmse:
            # 接受更新
            params = params_new
            residuals = residuals_new
            rmse = rmse_new
            history.append(rmse)
            lambda_param *= 0.5  # 减小阻尼
            
            if iteration % 5 == 0 or rmse < tol:
                print(f"  迭代 {iteration}: RMSE = {rmse:.6f} (λ={lambda_param:.2e})")
            
            # 收敛判断
            if abs(rmse - prev_rmse) < tol:
                print(f"  收敛于迭代 {iteration}!")
                break
            prev_rmse = rmse
        else:
            # 拒绝更新
            lambda_param *= 2.0  # 增大阻尼
            if iteration % 5 == 0:
                print(f"  迭代 {iteration}: 拒绝更新 (增大 λ={lambda_param:.2e})")
    
    # 解包最终参数
    K_opt, dist_opt, rvecs_opt, tvecs_opt = unpack_parameters(params, n_images)
    
    return K_opt, dist_opt, rvecs_opt, tvecs_opt, history, rmse

# 执行 LM 优化
print("=" * 50)
print("运行 Levenberg-Marquardt 优化...")
print("=" * 50)

K_opt, dist_opt, rvecs_opt, tvecs_opt, history, final_rmse = levenberg_marquardt(
    objpoints, imgpoints,
    K_initial, dist_initial, rvecs_initial, tvecs_initial,
    max_iter=50, lambda_init=1e-3, tol=1e-4
)

print(f"\n最终 RMSE: {final_rmse:.6f} 像素")

# 收敛曲线
plt.figure(figsize=(8, 4))
plt.plot(history, 'b-o', markersize=4)
plt.xlabel('迭代次数')
plt.ylabel('RMSE (像素)')
plt.title('LM 优化收敛曲线')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 第六步: 展示优化结果 =====

print("=" * 50)
print("优化结果")
print("=" * 50)

print(f"\n内参数矩阵 K (优化后):")
print(f"  fx = {K_opt[0, 0]:.4f} 像素")
print(f"  fy = {K_opt[1, 1]:.4f} 像素")
print(f"  cx = {K_opt[0, 2]:.4f} 像素")
print(f"  cy = {K_opt[1, 2]:.4f} 像素")

print(f"\n畸变系数:")
print(f"  k1 = {dist_opt[0, 0]:.6f}")
print(f"  k2 = {dist_opt[0, 1]:.6f}")
if len(dist_opt[0]) > 4:
    print(f"  p1 = {dist_opt[0, 2]:.6f}")
    print(f"  p2 = {dist_opt[0, 3]:.6f}")
    print(f"  k3 = {dist_opt[0, 4]:.6f}")

# 对比初始参数
print(f"\n对比初始值:")
print(f"  fx: {K_initial[0,0]:.2f} → {K_opt[0,0]:.2f} (变化: {abs(K_opt[0,0]-K_initial[0,0]):.2f})")
print(f"  fy: {K_initial[1,1]:.2f} → {K_opt[1,1]:.2f} (变化: {abs(K_opt[1,1]-K_initial[1,1]):.2f})")

# 计算最终残差分布
print(f"\n各图像重投影误差:")
for i in range(len(objpoints)):
    proj = project_points_custom(objpoints[i], rvecs_opt[i], tvecs_opt[i],
                                 K_opt, dist_opt.ravel())
    actual = imgpoints[i].reshape(-1, 2)
    err = np.sqrt(np.sum((proj - actual)**2, axis=1))
    print(f"  图像 {i+1}: 平均 = {np.mean(err):.4f}, 最大 = {np.max(err):.4f} 像素")

# 可视化: 绘制检测点 vs 投影点
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for idx in range(min(4, len(objpoints))):
    ax = axes[idx // 2, idx % 2]
    
    proj = project_points_custom(objpoints[idx], rvecs_opt[idx], tvecs_opt[idx],
                                 K_opt, dist_opt.ravel())
    actual = imgpoints[idx].reshape(-1, 2)
    
    ax.scatter(actual[:, 0], actual[:, 1], c='red', s=30, marker='x', label='检测点')
    ax.scatter(proj[:, 0], proj[:, 1], c='blue', s=30, marker='o', facecolors='none', label='投影点')
    
    for j in range(len(proj)):
        ax.plot([actual[j, 0], proj[j, 0]], [actual[j, 1], proj[j, 1]], 'k-', alpha=0.3)
    
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(f'图像 {idx+1}: 检测 vs 投影', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlabel('x (像素)')
    ax.set_ylabel('y (像素)')
plt.suptitle('角点检测 (红×) vs 优化后投影 (蓝○)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. 本章总结

### 核心知识点
1. **相机成像模型**：针孔模型、内参数矩阵、畸变模型
2. **张正友标定法**：使用平面棋盘格进行标定
   - 优点：只需一张棋盘格，鲁棒性好
   - 步骤：角点检测 → 初始估计 → 非线性优化
3. **Levenberg-Marquardt 算法**：非线性最小二乘优化
   - 结合高斯-牛顿法和梯度下降法
   - 自适应阻尼参数 λ
4. **重投影误差**：评估标定精度的核心指标

### 扩展练习
1. 使用手机拍摄自己的棋盘格图像进行标定
2. 对比使用不同张数标定图像的精度变化
3. 实现去畸变（使用标定结果校正畸变图像）
4. 尝试实现 Gauss-Newton 算法对比 LM 的性能

### 思考题
- 为什么需要至少 2 张不同角度的图像？
- LM 算法中 λ 参数的作用是什么？λ 的初始值如何选择？
- 有限差分法计算雅可比矩阵的精度受什么影响？


---

## 📝 练习：手写相机标定与畸变校正


**练习目标**：深入理解相机标定原理。

**要求**：
1. 实现内参矩阵的数学推导
2. 实现畸变模型的手写计算
3. 实现畸变校正的手写实现
4. 对比不同棋盘格图像数量的标定精度


**💡 小提示**：
- 使用 `cv_imread` / `cv_imwrite` 处理中文路径
- 除 OpenCV 读写函数外，其余代码全部手写
- 注意处理图像边界和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

def build_intrinsic_matrix(fx, fy, cx, cy):
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

def apply_distortion(point_3d, K, dist):
    X, Y, Z = point_3d[:3]
    xn, yn = X/Z, Y/Z
    r2 = xn**2 + yn**2
    k1, k2, p1, p2 = dist[:4]
    rd = 1 + k1*r2 + k2*r2**2
    xt = 2*p1*xn*yn + p2*(r2 + 2*xn**2)
    yt = p1*(r2 + 2*yn**2) + 2*p2*xn*yn
    xd = xn*rd + xt
    yd = yn*rd + yt
    fx, fy = K[0,0], K[1,1]
    return np.array([fx*xd + K[0,2], fy*yd + K[1,2], 1])

def undistort_manual(image, K, dist):
    h, w = image.shape[:2]
    mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K, (w, h), 5)
    return cv2.remap(image, mapx, mapy, cv2.INTER_LINEAR)

images = ['calib_image_1.jpg', 'calib_image_2.jpg', 'calib_image_3.jpg', 'calib_image_4.jpg']
existing = [f for f in images if os.path.exists(f)]
if len(existing) >= 2:
    board = (9, 6)
    objp = np.zeros((board[0]*board[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:board[0], 0:board[1]].T.reshape(-1, 2)
    obj_pts, img_pts = [], []
    for p in existing:
        img = cv_imread(p)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ret, corners = cv2.findChessboardCorners(gray, board, None)
        if ret:
            corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1),
                (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))
            obj_pts.append(objp)
            img_pts.append(corners2)
    ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(obj_pts, img_pts, gray.shape[::-1], None, None)
    print(f'内参矩阵:\n{K}')
    print(f'畸变系数: {dist.ravel()}')
    tp = np.array([0.5, 0.5, 5.0])
    px = apply_distortion(tp, K, dist.ravel())
    print(f'手写投影: {px[:2]}')
    img_t = cv_imread(existing[0])
    undist = undistort_manual(img_t, K, dist)
    cv_imwrite('undistorted.jpg', undist)
    print('畸变校正完成！')
else:
    print('标定板图像不足')



### 💻 代码要点解释

1. **数据准备**：加载测试图像，转换数据类型

2. **算法实现**：手写核心逻辑，逐步实现每个步骤

3. **对比验证**：与 OpenCV 对应函数结果进行数值对比

4. **结果可视化**：保存处理结果，观察效果差异

5. **扩展思考**：尝试不同参数，观察算法表现

---

</details>

---
